Import what you need

In [1]:
threads = 2

import os
os.environ['OMP_NUM_THREADS']=str(threads)
import tensorflow as tf

# PyTorch favours OMP_NUM_THREADS in environment
import torch

# Tensorflow needs explicit cofig calls
tf.config.threading.set_inter_op_parallelism_threads(threads)
tf.config.threading.set_intra_op_parallelism_threads(threads)

2025-11-11 07:26:32.109626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-11 07:26:32.157171: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-11 07:26:32.167934: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-11 07:26:32.226223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-11 07:26:33.276038: W tensorflow/compiler/tf2

In [2]:
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import glob

* Batch_size = write the same batch_size you used as **hps['batch_size']** 

In [3]:
X_test = tf.data.Dataset.load('01.datasets/intcoords/test')

# get batched version of dataset to feed to AAE model for prediction
X_test_batched = X_test.batch(64,drop_remainder=True)

# get numpy version for testing purposes
X_test_np = np.stack(list(X_test))

test_geom = np.moveaxis(np.stack(list(tf.data.Dataset.load('01.datasets/geoms/test'))),2,0)
test_geom.shape

mol_model = torch.jit.load('./03.model/features.pt')
torch_encoder = torch.jit.load('./03.model/encoder-norm.pt')
torch_decoder = torch.jit.load('./03.model/decoder-norm.pt')

2025-11-11 07:26:36.184485: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-11-11 07:26:36.842961: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
conf = './../dataset_Thermal-unfolding/trpcage_npt400_nH.pdb'
tr_full = md.load('./../dataset_Thermal-unfolding/trpcage_ds_nH.xtc',top=conf)

m = torch.jit.load('./03.model/model.pt')
lows = m(torch.tensor(tr_full.xyz)).numpy()

lows.shape

(50001, 2)

In [5]:
lows = np.loadtxt("lows.txt")

In [6]:
!pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)


In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Disentanglement metrics optimized for a 2D latent space (with robustness patches):
- nHSIC (normalized HSIC) with RBF kernel (median heuristic w/ fallback)
- Mutual Information (KSG-1) with k-NN (Chebyshev norm) + duplicate-aware jitter
- Reproducible subsampling and optional standardization
- Optimized for 2D analysis with dedicated visualizations

Outputs:
  1) <base>_metrics.csv           (key metrics)
  2) <base>_space.png/pdf         (scatter/hexbin plot z1 vs z2)
  3) <base>_marginals.png/pdf     (histograms of z1 and z2)
  4) <base>_dependence.png/pdf    (visualization of nHSIC and MI)

References:
- Gretton et al., "A Kernel Statistical Test of Independence", NIPS 2008
- Kraskov et al., "Estimating mutual information", PRE 2004
"""

from __future__ import annotations

import os
import csv
from typing import Dict, Tuple, Union

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

def set_global_seed(seed: int = 42) -> None:
    """Set global RNG seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # torch.use_deterministic_algorithms(True)  # uncomment if you need strict determinism

# ---------------------------------------------------------
# Efficient subsampling
# ---------------------------------------------------------

def subsample_rows(Z: Union[torch.Tensor, np.ndarray], max_n: int = 5000, seed: int = 42):
    """
    Subsample rows while preserving representativeness.
    For 2D, we keep it simple (random); stratification can be added if needed.
    Returns (Z_subsampled, n_original).
    """
    if isinstance(Z, torch.Tensor):
        n = Z.shape[0]
        if n <= max_n:
            return Z, n
        g = torch.Generator(device=Z.device).manual_seed(seed)
        idx = torch.randperm(n, generator=g, device=Z.device)[:max_n]
        return Z[idx], n

    elif isinstance(Z, np.ndarray):
        n = Z.shape[0]
        if n <= max_n:
            return Z, n
        rng = np.random.default_rng(seed)
        idx = rng.permutation(n)[:max_n]
        return Z[idx], n

    else:
        raise TypeError("Z must be a torch.Tensor or numpy.ndarray")

# ---------------------------------------------------------
# Robust standardization
# ---------------------------------------------------------

def zscore_columns(Z: Union[torch.Tensor, np.ndarray], robust: bool = False):
    """
    Z-score standardization (mean=0, std=1).
    If robust=True, use median and MAD instead of mean and std.
    """
    eps = 1e-12
    if isinstance(Z, torch.Tensor):
        if robust:
            median = Z.median(dim=0, keepdim=True).values
            mad = (Z - median).abs().median(dim=0, keepdim=True).values
            scale = 1.4826 * mad + eps  # MAD to std conversion
            return (Z - median) / scale
        else:
            mean = Z.mean(dim=0, keepdim=True)
            std = Z.std(dim=0, unbiased=False, keepdim=True) + eps
            return (Z - mean) / std

    elif isinstance(Z, np.ndarray):
        if robust:
            median = np.median(Z, axis=0, keepdims=True)
            mad = np.median(np.abs(Z - median), axis=0, keepdims=True)
            scale = 1.4826 * mad + eps
            return (Z - median) / scale
        else:
            mean = Z.mean(axis=0, keepdims=True)
            std = Z.std(axis=0, keepdims=True) + eps
            return (Z - mean) / std

    else:
        raise TypeError("Z must be a torch.Tensor or numpy.ndarray")

# ---------------------------------------------------------
# RBF kernel with optimized median heuristic (+ fallback)
# ---------------------------------------------------------

def _pairwise_sq_dists(x: torch.Tensor) -> torch.Tensor:
    # Computes the full squared Euclidean distance matrix
    x2 = (x ** 2).sum(1, keepdim=True)
    dist = x2 + x2.T - 2.0 * (x @ x.T)
    return torch.clamp(dist, min=0.0)


def rbf_kernel(x: torch.Tensor, sigma: torch.Tensor | None = None, subsample_for_bandwidth: bool = True):
    """
    RBF kernel with bandwidth selection via the median heuristic.
    For large datasets, sample to estimate the bandwidth.
    Returns (K, sigma).
    Fallback: if all pairwise distances are zero, use max(1.0, std(x)).
    """
    with torch.no_grad():
        n = x.shape[0]
        if sigma is None:
            if subsample_for_bandwidth and n > 2000:
                # Estimate bandwidth using a subset to avoid O(n^2) storage just for the median
                g = torch.Generator(device=x.device).manual_seed(0)
                idx = torch.randperm(n, generator=g, device=x.device)[:2000]
                xs = x[idx]
                dists_sample = _pairwise_sq_dists(xs)
                pos = dists_sample > 0
                if pos.any():
                    med = torch.median(dists_sample[pos])
                    sigma = torch.sqrt(med / 2.0 + 1e-12)
                else:
                    s = xs.std() + 1e-12
                    sigma = torch.maximum(torch.tensor(1.0, device=x.device), s)
            else:
                dists_full = _pairwise_sq_dists(x)
                pos = dists_full > 0
                if pos.any():
                    med = torch.median(dists_full[pos])
                    sigma = torch.sqrt(med / 2.0 + 1e-12)
                else:
                    s = x.std() + 1e-12
                    sigma = torch.maximum(torch.tensor(1.0, device=x.device), s)

        # Build full kernel once sigma is known
        dists = _pairwise_sq_dists(x)
        K = torch.exp(-dists / (2 * sigma ** 2 + 1e-12))

    return K, sigma

# ---------------------------------------------------------
# nHSIC (fixed centering)
# ---------------------------------------------------------

def nhsic_2d(z1: torch.Tensor, z2: torch.Tensor, return_sigma: bool = False):
    """
    Compute normalized HSIC between two 1D variables.
    Uses centered Gram matrices Kc = H K H and Lc = H L H.

    Returns:
        nhsic: float in [0, 1]
        (sigma1, sigma2): bandwidths used (if return_sigma=True)
    """
    assert z1.ndim == 2 and z2.ndim == 2
    assert z1.shape[1] == 1 and z2.shape[1] == 1

    n = z1.shape[0]
    H = torch.eye(n, device=z1.device) - torch.ones(n, n, device=z1.device) / n

    with torch.no_grad():
        K, sigma1 = rbf_kernel(z1)
        L, sigma2 = rbf_kernel(z2)

        Kc = H @ K @ H
        Lc = H @ L @ H

        num = torch.trace(Kc @ Lc)
        den = torch.sqrt(torch.trace(Kc @ Kc) * torch.trace(Lc @ Lc) + 1e-12)

        nhsic = (num / den).clamp(min=0.0, max=1.0).item()

    if return_sigma:
        return nhsic, (float(sigma1), float(sigma2))
    return nhsic

# ---------------------------------------------------------
# MI (KSG-1) optimized for 2D (+ duplicate-aware jitter)
# ---------------------------------------------------------

def _maybe_jitter(v: torch.Tensor, tol: float = 1e-12, sigma: float = 1e-8) -> torch.Tensor:
    """
    Add tiny Gaussian jitter if many pairs are (near-)duplicates.
    Trigger when >1% of pairwise absolute differences are < tol.
    """
    with torch.no_grad():
        dv = (v[:, None] - v[None, :]).abs()
        share_zeros = (dv < tol).float().mean()
        if share_zeros > 0.01:
            g = torch.Generator(device=v.device).manual_seed(0)
            v = v + torch.normal(0.0, sigma, size=v.shape, generator=g, device=v.device)
    return v


def mi_ksg_2d(z1: torch.Tensor, z2: torch.Tensor, k: int = 5) -> Tuple[float, float]:
    """
    Mutual Information via the KSG-1 estimator with Chebyshev (L-infinity) norm.

    Returns:
        mi_nats: float (MI in nats)
        mi_bits: float (MI in bits)
    """
    assert z1.ndim == 2 and z2.ndim == 2
    assert z1.shape[1] == 1 and z2.shape[1] == 1

    x = z1.squeeze(1)
    y = z2.squeeze(1)
    n = x.shape[0]

    if n < 3:
        raise ValueError("Need at least 3 samples to estimate MI with KSG.")
    # Keep k modest; practical guidance: 3–10, and not too large vs n
    if not (1 <= k <= max(1, n // 2)):
        raise ValueError(f"Invalid k={k} for n={n}. Suggested: 3–10 or ≤ n//2.")

    eps = 1e-10

    with torch.no_grad():
        # Defensive jitter if there are many duplicates/ties
        x = _maybe_jitter(x)
        y = _maybe_jitter(y)

        # Pairwise distances for Chebyshev norm
        dx = (x[:, None] - x[None, :]).abs()
        dy = (y[:, None] - y[None, :]).abs()
        dist_inf = torch.maximum(dx, dy)

        # Ignore self-distances
        inf_mask = torch.eye(n, device=x.device, dtype=torch.bool)
        dist_inf = dist_inf.masked_fill(inf_mask, float('inf'))

        # k-th nearest neighbor radius in joint space
        kth = torch.topk(dist_inf, k, largest=False).values[:, -1]
        eps_i = kth + eps  # small margin to avoid tie issues

        # Neighbor counts in marginals (exclude the point itself)
        nx = (dx < eps_i[:, None]).sum(dim=1) - 1
        ny = (dy < eps_i[:, None]).sum(dim=1) - 1

        psi = torch.digamma
        I = psi(torch.tensor(float(k), device=x.device)) \
            + psi(torch.tensor(float(n), device=x.device)) \
            - (psi(nx.float() + 1.0) + psi(ny.float() + 1.0)).mean()

        mi_nats = float(torch.clamp(I, min=0.0))
        mi_bits = mi_nats / np.log(2.0)

    return mi_nats, mi_bits

# ---------------------------------------------------------
# Descriptive statistics
# ---------------------------------------------------------

def compute_2d_statistics(Z_t: torch.Tensor) -> Dict[str, float]:
    """Compute descriptive statistics for a 2D latent space."""
    with torch.no_grad():
        n = Z_t.shape[0]
        z1 = Z_t[:, 0]
        z2 = Z_t[:, 1]
        z1_mean = float(z1.mean())
        z2_mean = float(z2.mean())
        z1_std = float(z1.std())
        z2_std = float(z2.std())

        # Safe Pearson correlation (avoid NaN when std≈0 or n<3)
        if n < 3 or z1_std < 1e-12 or z2_std < 1e-12:
            pearson = 0.0
        else:
            pearson = float(torch.corrcoef(Z_t.T)[0, 1])

        stats = {
            'z1_mean': z1_mean,
            'z1_std': z1_std,
            'z1_min': float(z1.min()),
            'z1_max': float(z1.max()),
            'z2_mean': z2_mean,
            'z2_std': z2_std,
            'z2_min': float(z2.min()),
            'z2_max': float(z2.max()),
            'pearson_corr': pearson,
        }
    return stats

# ---------------------------------------------------------
# 2D visualizations
# ---------------------------------------------------------

def plot_latent_space_2d(Z_t: torch.Tensor, out_base: str, title: str = "Latent Space 2D") -> None:
    """Hexbin scatter plot of the latent space with density coloring."""
    Z_np = Z_t.detach().cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 8))

    hb = ax.hexbin(Z_np[:, 0], Z_np[:, 1], gridsize=50, cmap='viridis',
                   mincnt=1, alpha=0.8, edgecolors='none')

    ax.set_xlabel('z₁', fontsize=14)
    ax.set_ylabel('z₂', fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Density', fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()


def plot_marginal_distributions(Z_t: torch.Tensor, out_base: str) -> None:
    """Histograms of marginal distributions."""
    Z_np = Z_t.detach().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for i, ax in enumerate(axes):
        ax.hist(Z_np[:, i], bins=50, alpha=0.7, color='steelblue',
                edgecolor='black', density=True)
        ax.set_xlabel('z₁' if i == 0 else 'z₂', fontsize=14)
        ax.set_ylabel('Density', fontsize=14)
        ax.set_title('Marginal Distribution z₁' if i == 0 else 'Marginal Distribution z₂',
                     fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()


def plot_dependence_metrics(nhsic: float, mi_nats: float, mi_bits: float, out_base: str) -> None:
    """Bar plot for dependence metrics."""
    fig, ax = plt.subplots(figsize=(10, 6))

    metrics = ['nHSIC', 'MI (nats)', 'MI (bits)']
    values = [float(nhsic), float(mi_nats), float(mi_bits)]

    bars = ax.barh(metrics, values, alpha=0.85, edgecolor='black', linewidth=1.2)

    # Add labels on bars
    vmax = max(values)
    pad = (0.02 * vmax) if vmax > 0 else 0.05
    for bar, val in zip(bars, values):
        width = bar.get_width()
        ax.text(width + pad + 1e-9, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', ha='left', va='center', fontsize=12, fontweight='bold')

    ax.set_xlabel('Value', fontsize=14)
    ax.set_title('Dependence Metrics (z₁, z₂)', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(0, max(vmax * 1.2, 1.0 if vmax == 0 else 0.1))  # avoid flat plot when all zeros

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()

# ---------------------------------------------------------
# CSV export
# ---------------------------------------------------------

def export_2d_metrics_csv(
    path: str,
    n_original: int,
    n_used: int,
    nhsic: float,
    mi_nats: float,
    mi_bits: float,
    stats: Dict[str, float],
    sigma_info: Tuple[float, float] | None = None,
    k_mi: int = 5,
) -> None:
    """Export all metrics to a CSV file."""
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])

        # Dataset info
        w.writerow(["n_original", n_original])
        w.writerow(["n_used", n_used])
        w.writerow(["n_latents", 2])

        # Estimator settings
        w.writerow(["k_mi", k_mi])
        w.writerow(["mi_unit_primary", "nats"])
        w.writerow(["mi_unit_secondary", "bits"])

        # Dependence metrics
        w.writerow(["nhsic_z1_z2", f"{nhsic:.6f}"])
        w.writerow(["mi_z1_z2_nats", f"{mi_nats:.6f}"])
        w.writerow(["mi_z1_z2_bits", f"{mi_bits:.6f}"])

        # Descriptive statistics
        for key, val in stats.items():
            w.writerow([key, f"{val:.6f}"])

        # Bandwidth info (if available)
        if sigma_info is not None:
            w.writerow(["rbf_sigma_z1", f"{sigma_info[0]:.6f}"])
            w.writerow(["rbf_sigma_z2", f"{sigma_info[1]:.6f}"])

# ---------------------------------------------------------
# Main 2D analysis pipeline
# ---------------------------------------------------------

def analyze_2d_latent_space(
    Z: Union[np.ndarray, torch.Tensor],
    out_dir: str = ".",
    device: str | None = None,
    standardize: bool = True,
    robust_standardize: bool = False,
    max_n: int = 5000,
    seed: int = 42,
    k_mi: int = 5,
    base_name: str = "latent_2d",
    enforce_memory_guard: bool = True,
) -> Dict[str, object]:
    """
    Full pipeline for analyzing a 2D latent space.

    Args:
        Z: array/tensor [n, 2] - 2D latent space
        out_dir: output directory
        device: 'cpu', 'cuda', or None (auto)
        standardize: apply z-score
        robust_standardize: use median/MAD instead of mean/std
        max_n: max samples for subsampling
        seed: global seed for reproducibility
        k_mi: k-nearest neighbors for MI estimator
        base_name: prefix for output files
        enforce_memory_guard: prevent OOM with very large n

    Returns:
        dict with metrics and file paths
    """
    os.makedirs(out_dir, exist_ok=True)
    set_global_seed(seed)

    # Validate input
    if not isinstance(Z, (np.ndarray, torch.Tensor)):
        raise TypeError("Z must be a numpy.ndarray or torch.Tensor")
    if Z.ndim != 2 or Z.shape[1] != 2:
        raise ValueError(f"Z must have shape [n, 2], found {tuple(Z.shape)}")
    if Z.shape[0] < 3:
        raise ValueError("Need at least 3 samples for MI/KSG and correlation.")

    print("[INFO] 2D latent space analysis")
    print(f"[INFO] Original shape: {Z.shape}")

    # 1) Standardization (optional)
    if standardize:
        Z = zscore_columns(Z, robust=robust_standardize)
        print(f"[INFO] Standardization: {'robust (MAD)' if robust_standardize else 'classical (std)'}")

    # 2) Subsampling
    Z_sub, n_original = subsample_rows(Z, max_n=max_n, seed=seed)
    print(f"[INFO] Subsampling: {n_original} → {Z_sub.shape[0]} samples")

    # Memory guard for O(n^2) ops used in HSIC & distances
    if enforce_memory_guard and Z_sub.shape[0] > 8000:
        raise MemoryError("Too many samples after subsampling for O(n^2) ops. "
                          "Please reduce max_n (e.g., ≤ 8000).")

    # 3) Convert to torch
    if isinstance(Z_sub, np.ndarray):
        Z_t = torch.from_numpy(Z_sub)
    else:
        Z_t = Z_sub
    Z_t = Z_t.float()

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    Z_t = Z_t.to(device)
    print(f"[INFO] Device: {device}")

    n_used = Z_t.shape[0]

    # 4) Metrics
    print("[INFO] Computing nHSIC…")
    nhsic, sigma_info = nhsic_2d(Z_t[:, 0:1], Z_t[:, 1:2], return_sigma=True)

    print(f"[INFO] Computing MI (KSG-1, k={k_mi})…")
    mi_nats, mi_bits = mi_ksg_2d(Z_t[:, 0:1], Z_t[:, 1:2], k=k_mi)

    print("[INFO] Computing descriptive statistics…")
    stats = compute_2d_statistics(Z_t)

    # 5) CSV export
    csv_path = os.path.join(out_dir, f"{base_name}_metrics.csv")
    export_2d_metrics_csv(
        csv_path, n_original, n_used, nhsic, mi_nats, mi_bits, stats, sigma_info, k_mi=k_mi
    )
    print(f"[EXPORT] Metrics: {csv_path}")

    # 6) Visualizations
    print("[INFO] Generating plots…")

    latent_base = os.path.join(out_dir, f"{base_name}_space")
    plot_latent_space_2d(Z_t, latent_base)
    print(f"[EXPORT] Latent space: {latent_base}.png/pdf")

    marginal_base = os.path.join(out_dir, f"{base_name}_marginals")
    plot_marginal_distributions(Z_t, marginal_base)
    print(f"[EXPORT] Marginals: {marginal_base}.png/pdf")

    metrics_base = os.path.join(out_dir, f"{base_name}_dependence")
    plot_dependence_metrics(nhsic, mi_nats, mi_bits, metrics_base)
    print(f"[EXPORT] Dependence metrics: {metrics_base}.png/pdf")

    # 7) Summary
    print("\n" + "=" * 60)
    print("2D ANALYSIS RESULTS")
    print("=" * 60)
    print(f"nHSIC(z₁, z₂):     {nhsic:.6f}  [0=independent, 1=dependent]")
    print(f"MI(z₁, z₂) nats:   {mi_nats:.6f}")
    print(f"MI(z₁, z₂) bits:   {mi_bits:.6f}")
    print(f"Pearson corr:      {stats['pearson_corr']:.6f}")
    print(f"k (KSG-1):         {k_mi}")
    print("=" * 60 + "\n")

    return {
        "nhsic": nhsic,
        "mi_nats": mi_nats,
        "mi_bits": mi_bits,
        "sigma_z1": sigma_info[0],
        "sigma_z2": sigma_info[1],
        "statistics": stats,
        "csv_path": csv_path,
        "n_original": n_original,
        "n_used": n_used,
        "k_mi": k_mi,
    }

In [8]:
res = analyze_2d_latent_space(
    lows,                        # your 2D latent array or tensor, shape [n, 2]
    out_dir=".",              # output folder for CSV and plots
    device='cpu',              # 'cuda', 'cpu', or None for auto-detect
    standardize=True,         # apply z-score standardization
    robust_standardize=False, # use median/MAD instead of mean/std if True
    max_n=8000,               # max number of samples to keep (subsampling)
    seed=42,                  # random seed for reproducibility
    k_mi=5,                   # k for KSG-1 MI estimator
    base_name="latent_2d_demo", # prefix for output file names
    enforce_memory_guard=True, # safety check for very large datasets
)


[INFO] 2D latent space analysis
[INFO] Original shape: (50001, 2)
[INFO] Standardization: classical (std)
[INFO] Subsampling: 50001 → 8000 samples
[INFO] Device: cpu
[INFO] Computing nHSIC…
[INFO] Computing MI (KSG-1, k=5)…
[INFO] Computing descriptive statistics…
[EXPORT] Metrics: ./latent_2d_demo_metrics.csv
[INFO] Generating plots…
[EXPORT] Latent space: ./latent_2d_demo_space.png/pdf
[EXPORT] Marginals: ./latent_2d_demo_marginals.png/pdf
[EXPORT] Dependence metrics: ./latent_2d_demo_dependence.png/pdf

2D ANALYSIS RESULTS
nHSIC(z₁, z₂):     0.005954  [0=independent, 1=dependent]
MI(z₁, z₂) nats:   0.206918
MI(z₁, z₂) bits:   0.298519
Pearson corr:      -0.014813
k (KSG-1):         5

